In [15]:
import pandas as pd
import numpy as np
df=pd.read_csv("C:\\Users\\hp\\Documents\\ai-engineering-projects\\week1\\data\\application_train.csv")


In [16]:
print(f"Original shape: {df.shape}")
print(f"\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))


Original shape: (307511, 122)

Missing values:
COMMONAREA_MEDI             214865
COMMONAREA_AVG              214865
COMMONAREA_MODE             214865
NONLIVINGAPARTMENTS_MEDI    213514
NONLIVINGAPARTMENTS_MODE    213514
                             ...  
EXT_SOURCE_2                   660
AMT_GOODS_PRICE                278
AMT_ANNUITY                     12
CNT_FAM_MEMBERS                  2
DAYS_LAST_PHONE_CHANGE           1
Length: 67, dtype: int64


In [17]:
# Age (numeric)
df['DAYS_BIRTH'].fillna(df['DAYS_BIRTH'].median(), inplace=True)

# Income (numeric)
df['AMT_INCOME_TOTAL'].fillna(df['AMT_INCOME_TOTAL'].median(), inplace=True)

# Employment days (numeric)
df['DAYS_EMPLOYED'].fillna(df['DAYS_EMPLOYED'].median(), inplace=True)

# Drop rows where target is missing
df = df[df['TARGET'].notna()]

print(f"After handling missing values:")
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")


C:\Users\hp\AppData\Local\Temp\ipykernel_18588\2580113790.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['DAYS_BIRTH'].fillna(df['DAYS_BIRTH'].median(), inplace=True)
C:\Users\hp\AppData\Local\Temp\ipykernel_18588\2580113790.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy

After handling missing values:
Shape: (307511, 122)
Missing values: 9152465


In [18]:
# Convert age from days to years
df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365.25

# Fix employment (some have negative days = error)
df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)  # Remove obvious error
df['DAYS_EMPLOYED'].fillna(df['DAYS_EMPLOYED'].median(), inplace=True)

print(f"Age range: {df['AGE_YEARS'].min():.0f} to {df['AGE_YEARS'].max():.0f}")
print(f"Employment days range: {df['DAYS_EMPLOYED'].min():.0f} to {df['DAYS_EMPLOYED'].max():.0f}")


Age range: 21 to 69
Employment days range: -17912 to 0


C:\Users\hp\AppData\Local\Temp\ipykernel_18588\3200640731.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)  # Remove obvious error


In [19]:
df['CODE_GENDER'] = (df['CODE_GENDER'] == 'M').astype(int)

# Ownership flag
df['FLAG_OWN_CAR'] = (df['FLAG_OWN_CAR'] == 'Y').astype(int)
df['FLAG_OWN_REALTY'] = (df['FLAG_OWN_REALTY'] == 'Y').astype(int)

print("Encoded categorical variables")
print(df[['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']].head())


Encoded categorical variables
   CODE_GENDER  FLAG_OWN_CAR  FLAG_OWN_REALTY
0            1             0                1
1            0             0                0
2            1             1                1
3            0             0                1
4            1             0                1


In [20]:
# Remove customers with unrealistic income
df = df[df['AMT_INCOME_TOTAL'] < 1000000]

# Remove very young/old customers
df = df[(df['AGE_YEARS'] >= 18) & (df['AGE_YEARS'] <= 100)]

print(f"After removing outliers:")
print(f"Shape: {df.shape}")


After removing outliers:
Shape: (307261, 123)


In [21]:
features_to_keep = [
    'TARGET',
    'AGE_YEARS',
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'CNT_CHILDREN',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_INCOME_TYPE',
    'OCCUPATION_TYPE'
]

# Keep only these columns
df_clean = df[features_to_keep].copy()

print(f"Selected {len(features_to_keep)} features")
print(f"Shape: {df_clean.shape}")


Selected 13 features
Shape: (307261, 13)


In [22]:
categorical_cols = ['NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE']

for col in categorical_cols:
    # Create dummy variables
    dummies = pd.get_dummies(df_clean[col], prefix=col, drop_first=True)
    df_clean = pd.concat([df_clean, dummies], axis=1)
    # Drop original
    df_clean = df_clean.drop(col, axis=1)

print(f"After encoding:")
print(f"Shape: {df_clean.shape}")
print(f"Columns: {df_clean.columns.tolist()}")


After encoding:
Shape: (307261, 26)
Columns: ['TARGET', 'AGE_YEARS', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'OCCUPATION_TYPE', 'NAME_EDUCATION_TYPE_Higher education', 'NAME_EDUCATION_TYPE_Incomplete higher', 'NAME_EDUCATION_TYPE_Lower secondary', 'NAME_EDUCATION_TYPE_Secondary / secondary special', 'NAME_FAMILY_STATUS_Married', 'NAME_FAMILY_STATUS_Separated', 'NAME_FAMILY_STATUS_Single / not married', 'NAME_FAMILY_STATUS_Unknown', 'NAME_FAMILY_STATUS_Widow', 'NAME_INCOME_TYPE_Commercial associate', 'NAME_INCOME_TYPE_Maternity leave', 'NAME_INCOME_TYPE_Pensioner', 'NAME_INCOME_TYPE_State servant', 'NAME_INCOME_TYPE_Student', 'NAME_INCOME_TYPE_Unemployed', 'NAME_INCOME_TYPE_Working']


In [24]:
df_clean.to_csv('C:\\Users\\hp\\Documents\\ai-engineering-projects\\week1\\data\\application_train_cleaned.csv', index=False)
print("✅ Cleaned data saved to: data/application_train_cleaned.csv")
print(f"Shape: {df_clean.shape}")
print(f"\nData types:")
print(df_clean.dtypes)

✅ Cleaned data saved to: data/application_train_cleaned.csv
Shape: (307261, 26)

Data types:
TARGET                                                 int64
AGE_YEARS                                            float64
AMT_INCOME_TOTAL                                     float64
AMT_CREDIT                                           float64
AMT_ANNUITY                                          float64
CODE_GENDER                                            int64
FLAG_OWN_CAR                                           int64
FLAG_OWN_REALTY                                        int64
CNT_CHILDREN                                           int64
OCCUPATION_TYPE                                       object
NAME_EDUCATION_TYPE_Higher education                    bool
NAME_EDUCATION_TYPE_Incomplete higher                   bool
NAME_EDUCATION_TYPE_Lower secondary                     bool
NAME_EDUCATION_TYPE_Secondary / secondary special       bool
NAME_FAMILY_STATUS_Married                           

In [25]:
print("="*60)
print("DATA CLEANING SUMMARY")
print("="*60)
print(f"Original shape: 307511 rows, 122 columns")
print(f"Cleaned shape: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
print(f"Missing values: 0 (all handled)")
print(f"Categorical variables: All encoded to numbers")
print(f"Outliers: Removed (unrealistic income, invalid ages)")
print("="*60)


DATA CLEANING SUMMARY
Original shape: 307511 rows, 122 columns
Cleaned shape: 307261 rows, 26 columns
Missing values: 0 (all handled)
Categorical variables: All encoded to numbers
Outliers: Removed (unrealistic income, invalid ages)
